The environment configuration
=============================

`PPTopoGym` is configured by exactly one argument: a plain dictionary, `env_config`.
Everything the environment does -- which grid it solves, which actions an agent may take,
what an observation contains, how a step is rewarded and how expensive a step is -- is
decided by keys in that dictionary.

This notebook walks through **every** key: what it does, what it defaults to, and what
changes when you set it. It is the reference companion to `getting_started.ipynb`, which
shows the environment in use rather than the knobs it exposes.

Two properties of `env_config` are worth stating up front:

* **It must stay serializable.** RLlib pickles the config to send it to its workers, so the
  values you put in it have to survive a `pickle` round trip. Module-level functions are
  fine; lambdas and closures are not.
* **It is copied, not borrowed.** `PPTopoGym` deep-copies the net (sharing only the
  read-only profile tables) so that two environments built from the same config never share
  a mutable grid. The snapshot is kept as `env.orig_config`, which is the cheap way to give
  an agent its own environment.

In [1]:
import time

import numpy as np
import pandas as pd
from gymnasium import spaces

from pandapower_env.data.example_configs import config_case30
from pandapower_env.environments.simulation_env import PPTopoGym
from pandapower_env.observation_space.obs_space_utils import (
    build_info_observation_registry,
    build_observation_registry,
)

1. What a configuration looks like
----------------------------------

`pandapower_env.data.example_configs` assembles ready-made configurations. Building one is
the expensive part of this notebook: it loads the grid, scales the load/generation
timeseries until lines overload, expands every substation into a double busbar and then
verifies each generated action with a throwaway power flow.

Build it **once** and reuse it -- that is what `env.orig_config` is for.

In [2]:
started = time.time()
config = config_case30()
print(f"config_case30() took {time.time() - started:.1f} s")
print("keys:", list(config))

config_case30() took 5.5 s
keys: ['net', 'n_episodes', 'episode_length', 'action_space', 'nminus1']


Only four keys are **required**; `config_case30` sets one optional key (`nminus1`) on top.
Everything else in this notebook is optional and has a default.

| required key | type | meaning |
|---|---|---|
| `net` | `pandapowerNet` | the grid, already expanded into double-busbar substations |
| `n_episodes` | `int` | how many episodes the timeseries is cut into |
| `episode_length` | `int` | timesteps per episode |
| `action_space` | `list[dict]` | the action specification, turned into `env.df_actions` |

A net can also be loaded from disk instead of passed as an object -- put the path in
`net_file`. Note that the `net` key is still read first, so pass `net=None` alongside it.

In [3]:
net = config["net"]
print(net)

This pandapower network includes the following parameter tables:
   - bus (93 elements)
   - load (20 elements)
   - gen (5 elements)
   - switch (116 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)
   - multi_bb_substation (10 elements)
 and the following results tables:
   - res_bus (30 elements)
   - res_line (41 elements)
   - res_ext_grid (1 element)
   - res_load (20 elements)
   - res_shunt (2 elements)
   - res_gen (5 elements)


2. `action_space` -- what the agent may do
------------------------------------------

The list of action dictionaries is turned into a DataFrame (`env.df_actions`) at
construction time, and `env.action_space` is a `Discrete` over its rows.

The columns drive `load_action`:

* `open_switches` / `closed_switches` -- the busbar configuration of a substation,
* `lines` / `disconnect_lines` -- connect or disconnect a line,
* `trafos` / `tap_pos` -- move a phase-shift transformer tap (see `config_30pst`).

**Row 0 is always DoNothing** and is special-cased: it keeps the previous topology and the
previous `converged` flag, so it is not "an action that happens to change nothing".

In [4]:
env = PPTopoGym(config)
print("action space:", env.action_space)
env.df_actions.head()

action space: Discrete(289)


,action,substations,states,open_switches,closed_switches,lines,disconnect_lines
0,0,[],[],[],[],[],[]
1,1,[0],[000000],[],"[1, 3, 5, 7, 9, 11, 2, 4, 6, 8, 10, 12, 0]",[],[]
2,2,[0],[000101],"[0, 7, 11, 2, 4, 6, 10]","[1, 3, 5, 9, 8, 12]",[],[]
3,3,[0],[000110],"[0, 7, 9, 2, 4, 6, 12]","[1, 3, 5, 11, 8, 10]",[],[]
4,4,[0],[000111],"[0, 7, 9, 11, 2, 4, 6]","[1, 3, 5, 8, 10, 12]",[],[]


3. `n_episodes`, `episode_length`, `resolution`
-----------------------------------------------

The timeseries is a flat table of timesteps; `episode_length` cuts it into episodes and
`n_episodes` says how many of them exist. `resolution` is the length of one timestep **in
hours** and is only used where a power (MW) has to become an energy (MWh) -- the overload
energy in the reward and in the info dict.

`reset()` has non-default gym semantics here: pass `options={"index": N}` to start at a
specific timestep. Without it, `reset()` picks a *random* episode start, which is what you
want for training and not what you want for a reproducible notebook.

In [5]:
print("timesteps available:", env.n_total_timesteps)
print("episode_length:", env.episode_length, " n_episodes:", env.n_episodes)
print("resolution:", env.resolution, "h per timestep")

observation, info = env.reset(options={"index": 4000})
print("reset to profile index:", env.index)

timesteps available: 35136
episode_length: 96  n_episodes: 366
resolution: 1.0 h per timestep
reset to profile index: 4000


4. Where the timeseries comes from: `net.profiles` vs. `profiles`
-----------------------------------------------------------------

There are two ways to drive the injections, and they differ in one important respect.

* **`net.profiles`** (the Simbench route) holds *per-unit shapes*. `setup_profiles` scales
  them by the net's base `p_mw` / `q_mvar` / `vm_pu`.
* **`env_config["profiles"]`** holds *absolute values*, one column per element, and is
  **assigned, not multiplied**. It wins over `net.profiles` when present.

Both end up in the same six derived tables (`env.df_profiles_*`). Mixing the two up
double-scales the grid without raising anything, so the rule is: config profiles are
already-scaled numbers.

The config route is the way to build a **frozen episode** -- one profile row repeated for
the whole episode, so that every timestep poses the same physical problem.

In [6]:
print("net.profiles:", {key: df.shape for key, df in net.profiles.items()})
print("derived table df_profiles_load_p:", env.df_profiles_load_p.shape)

frozen_index, frozen_length = 4000, 8


def repeat_row(profile_table: pd.DataFrame) -> pd.DataFrame:
    """Repeat one timestep of a derived profile table into a constant episode."""
    row = profile_table.iloc[[frozen_index]]
    return pd.concat([row] * frozen_length, ignore_index=True)


frozen_config = dict(config)
frozen_config["episode_length"] = frozen_length
frozen_config["profiles"] = {
    "load": {"p_mw": repeat_row(env.df_profiles_load_p), "q_mvar": repeat_row(env.df_profiles_load_q)},
    "gen": {"p_mw": repeat_row(env.df_profiles_gen_p), "vm_pu": repeat_row(env.df_profiles_gen_vm)},
}

frozen_env = PPTopoGym(frozen_config)
frozen_env.reset(options={"index": 0})
rewards = [frozen_env.step(0)[1] for _ in range(3)]
print("DoNothing rewards on a frozen episode:", rewards)

net.profiles: {'load': (35136, 41), 'powerplants': (35136, 6), 'renewables': (35136, 1)}
derived table df_profiles_load_p: (35136, 20)


DoNothing rewards on a frozen episode: [np.float64(144.47563254709584), np.float64(144.47563254709584), np.float64(144.47563254709584)]


Only the six injection pairs above are accepted -- `load.p_mw`, `load.q_mvar`,
`gen.p_mw`, `gen.vm_pu`, `sgen.p_mw`, `sgen.q_mvar`. Anything else raises rather than being
silently dropped, because a timeseries that is quietly ignored is exactly the failure this
route exists to prevent.

In [7]:
bad_config = dict(config)
bad_config["profiles"] = {"line": {"max_i_ka": pd.DataFrame({0: [1.0, 1.0]})}}
try:
    PPTopoGym(bad_config)
except RuntimeError as error:
    print("RuntimeError:", str(error)[:120], "...")

RuntimeError: Timeseries for 'line.max_i_ka' is not supported on this branch: only [('gen', 'p_mw'), ('gen', 'vm_pu'), ('load', 'p_mw' ...


5. The power flow: `pf_type` and `backend`
------------------------------------------

`pf_type` selects `"ac"` (default) or `"dc"`. `backend` selects the solver behind
`run_pf`:

* `"pandapower"` (default) -- pandapower's own power flow, accelerated by lightsim2grid
  where it can be (`use_ls2g="auto"`). Every stored result in this repository is
  bit-identical to this path.
* `"lightsim"` -- re-expresses the switched topology as a switch-free *mirror net* with
  per-element bus assignments and solves it directly in lightsim2grid. Roughly an order of
  magnitude faster on the power flow itself, and it covers the N-1 sweep too, but it agrees
  to ~1e-11 rather than bit-for-bit, so it is opt-in.

In [8]:
def time_steps(env_config: dict, n_steps: int = 5) -> float:
    """Median seconds per step of a DoNothing/switching mix on a fresh environment."""
    timed_env = PPTopoGym(env_config)
    timed_env.reset(options={"index": 4000})
    timed_env.step(0)  # warm-up
    durations = []
    for step in range(n_steps):
        timed_env.reset(options={"index": 4000})
        started = time.time()
        timed_env.step(5 if step % 2 else 0)
        durations.append(time.time() - started)
    return float(np.median(durations))


lightsim_config = dict(config)
lightsim_config["backend"] = "lightsim"
print(f"backend='pandapower': {time_steps(config) * 1e3:6.2f} ms/step")
print(f"backend='lightsim'  : {time_steps(lightsim_config) * 1e3:6.2f} ms/step")

backend='pandapower':  15.32 ms/step


backend='lightsim'  :   4.89 ms/step


/mnt/home/dkoehler/all_projects/pandapower-env/.venv/lib64/python3.12/site-packages/lightsim2grid/gridmodel/from_pandapower/_aux_add_slack.py:60: UserWarning: LightSim has not found any generators tagged as "slack bus" in the pandapower network.I will attempt to add some from the ext_grid.
  warnings.warn("LightSim has not found any generators tagged as \"slack bus\" in the pandapower network."


6. N-1 security: `nminus1`, `n-1-topk`, `n-1 parallel`, `n-1 workers`
---------------------------------------------------------------------

With `nminus1=True` every step additionally sweeps the single-outage contingencies and
writes `max_loading_percent` / `min_loading_percent` plus the `cause_*` columns into
`res_line` / `res_trafo`. This is by far the most expensive key in the configuration --
one power flow per contingency instead of one per step.

Three keys make it affordable:

* `n-1-topk` (default `100.0`) -- only the top *k* % of lines by N-0 apparent power flow
  are switched off. It cuts the *contingency* set, not the *monitored* set: loadings are
  still reported for every line, just over fewer outages, so the reported risk can only go
  down. Trafo contingencies are not filtered.
* `n-1 parallel` (default `False`) -- spread the contingencies over `loky` worker
  processes. Bit-for-bit identical to the serial sweep. It **auto-degrades to serial inside
  child processes**, so it is a no-op in spawned RL workers and in the parallel greedy
  agent; it only helps the main process.
* `n-1 workers` (default: all CPUs) -- how many workers. Useful workers scale with the
  *contingency count*, roughly contingencies / 6, not with the core count. Over-provisioning
  on a small grid makes it slower, so cap it.

In [9]:
nminus1_config = dict(config)
nminus1_config["nminus1"] = True

nminus1_env = PPTopoGym(nminus1_config)
nminus1_env.reset(options={"index": 4000})
started = time.time()
nminus1_env.step(0)
nminus1_step = time.time() - started
print(f"N-1 step: {nminus1_step * 1e3:.0f} ms "
      f"({nminus1_step / time_steps(config):.0f}x a plain step)")
print("extra res_line columns:",
      [column for column in nminus1_env.net.res_line.columns
       if column not in env.net.res_line.columns])

N-1 step: 557 ms (39x a plain step)
extra res_line columns: ['causes_overloading', 'cause_element', 'cause_index', 'max_loading_percent', 'min_loading_percent']


In [10]:
topk_config = dict(nminus1_config)
topk_config["n-1-topk"] = 25.0

topk_env = PPTopoGym(topk_config)
topk_env.reset(options={"index": 4000})
started = time.time()
topk_env.step(0)
print(f"N-1 step with n-1-topk=25: {(time.time() - started) * 1e3:.0f} ms")
print("worst reported loading, full N-1:",
      round(nminus1_env.net.res_line["max_loading_percent"].max(), 3))
print("worst reported loading, top 25%:",
      round(topk_env.net.res_line["max_loading_percent"].max(), 3))

N-1 step with n-1-topk=25: 184 ms
worst reported loading, full N-1: 111.292
worst reported loading, top 25%: 111.292


On case30 the two agree: the outage that drives the worst post-contingency loading is on a
line that is already among the most loaded, so the cheaper sweep finds it too. That is the
common case and the reason the filter is useful -- but it is not guaranteed, which is why
the default is the full sweep.

7. The reward: `reward`, `worst_reward`, `clip_max_loading`
------------------------------------------------------------

The default reward is `clip_max_loading - clip(max line loading, 0, clip_max_loading)`, so
it is large when the grid is relieved and `0` once a line is at or beyond
`clip_max_loading` (default `200.0`).

`reward` overrides it, and takes either

* a **string** naming a function in `pandapower_env/data/rewards.py`, or
* any **callable** taking the environment and returning a float.

The callable form is the reason the config has to stay picklable: use a module-level
function, not a lambda, if the config ever reaches an RLlib worker.

`worst_reward` (default `-1000.0`) is what a step returns when the power flow does not
converge. That case is **not an exception**: the step returns `terminated=True`,
`info["crashed"]=True` and an all-zero observation, so check `info` rather than trusting
the observation.

In [11]:
def mean_loading_reward(environment: PPTopoGym) -> float:
    """Custom reward: the negative mean line loading of the solved grid."""
    return float(-environment.net.res_line["loading_percent"].mean())


for reward in [None, "reward_normalized", mean_loading_reward]:
    reward_config = dict(config)
    if reward is not None:
        reward_config["reward"] = reward
    reward_env = PPTopoGym(reward_config)
    reward_env.reset(options={"index": 4000})
    name = "default" if reward is None else getattr(reward, "__name__", reward)
    print(f"{name:>24}: {reward_env.step(0)[1]:.4f}")

                 default: 144.4756


       reward_normalized: 0.9448


     mean_loading_reward: -16.6173


8. Observations: `observation_keys`, `fix_obs_space`, `static_obs_space`, `observation`
----------------------------------------------------------------------------------------

By default the environment emits **every** observation in the registry. That is a lot of
data per step, and the set of keys fixes the input dimension of anything you train, so
narrowing it with `observation_keys` is usually the first thing to do.

In [12]:
registry = build_observation_registry()
print(f"{len(registry)} observations are emitted by default:")
print(sorted(registry))

43 observations are emitted by default:
['adjacency_matrix', 'bus_lookup_table', 'bus_voltage_angle', 'bus_voltage_magnitude', 'gen_bus', 'gen_power_p_mw_profile', 'gen_power_p_mw_runpf', 'gen_status', 'gen_vm_pu_profile', 'gen_vm_pu_runpf', 'line_from_bus', 'line_loadings', 'line_power_flow_p_mw', 'line_power_flow_q_mvar', 'line_status', 'line_thermal_limit', 'line_to_bus', 'load_bus', 'load_power_p_mw_profile', 'load_power_p_mw_runpf', 'load_power_q_mvar_profile', 'load_power_q_mvar_runpf', 'load_status', 'node_slot_map', 'sgen_bus', 'sgen_power_p_mw_profile', 'sgen_power_p_mw_runpf', 'sgen_power_q_mvar_profile', 'sgen_power_q_mvar_runpf', 'sgen_status', 'switch_positions', 'system_losses', 'total_power_demand_profile', 'total_power_demand_runpf', 'total_power_generation_profile', 'total_power_generation_runpf', 'trafo_hv_bus', 'trafo_lv_bus', 'transformer_loading_percent', 'transformer_power_flow_p_mw', 'transformer_power_flow_q_mvar', 'transformer_status', 'transformer_tap_position

In [13]:
narrow_config = dict(config)
narrow_config["observation_keys"] = ["line_loadings", "bus_voltage_magnitude", "adjacency_matrix"]

narrow_env = PPTopoGym(narrow_config)
narrow_observation, _ = narrow_env.reset(options={"index": 4000})
print({key: value.shape for key, value in narrow_observation.items()})

{'adjacency_matrix': (41, 2), 'bus_voltage_magnitude': (30,), 'line_loadings': (41,)}


`fix_obs_space` (default `True`) decides the *length* of the bus-mapped observations. With
it on, table observations are aggregated to the electrical nodes; with it off, they keep
the full pandapower table length -- and that table is much longer than the grid, because
expanding substations into double busbars adds many out-of-service auxiliary buses
(case30: 93 bus rows for ~30 active nodes).

In [14]:
wide_config = dict(narrow_config)
wide_config["fix_obs_space"] = False

wide_observation, _ = PPTopoGym(wide_config).reset(options={"index": 4000})
print("fix_obs_space=True  -> bus_voltage_magnitude:",
      narrow_observation["bus_voltage_magnitude"].shape)
print("fix_obs_space=False -> bus_voltage_magnitude:",
      wide_observation["bus_voltage_magnitude"].shape)
print("net.bus rows:", len(net.bus))

fix_obs_space=True  -> bus_voltage_magnitude: (30,)
fix_obs_space=False -> bus_voltage_magnitude: (93,)
net.bus rows: 93


`static_obs_space` (default `False`) fixes a subtler problem. The node count **grows when a
substation splits** -- case30 goes from 30 nodes at reset to as many as 35 over a sequence
of actions -- while `observation_space` was declared at the reset topology. So the declared
space is a lie by default: most observations fail `observation_space.contains(obs)`, and
`SyncVectorEnv` / `AsyncVectorEnv` crash with a broadcast error.

Turning it on declares node observations at their static upper bound and zero-pads up to
it. It is off by default only because switching it on changes observation shapes and would
break a network trained against the old ones. Rewards are identical either way.

In [15]:
split_action = int(env.df_actions.index[5])

for static in (False, True):
    contract_config = dict(narrow_config)
    contract_config["static_obs_space"] = static
    contract_env = PPTopoGym(contract_config)
    contract_env.reset(options={"index": 4000})
    observation = contract_env.step(split_action)[0]
    print(f"static_obs_space={static!s:<5} "
          f"bus_voltage_magnitude={observation['bus_voltage_magnitude'].shape} "
          f"contains(obs)={contract_env.observation_space.contains(observation)}")

static_obs_space=False bus_voltage_magnitude=(31,) contains(obs)=False


static_obs_space=True  bus_voltage_magnitude=(40,) contains(obs)=True


`observation` adds observations the registry does not have. Each entry is a dict with a
`name`, a `function` taking the environment, and a gymnasium `spaces` entry describing what
that function returns.

In [16]:
def mean_loading_observation(environment: PPTopoGym) -> np.ndarray:
    """Custom observation: the mean line loading, as a length-1 float32 array."""
    return np.array([environment.net.res_line["loading_percent"].mean()], dtype=np.float32)


custom_config = dict(config)
custom_config["observation_keys"] = ["line_loadings"]
custom_config["observation"] = [{
    "name": "mean_loading",
    "function": mean_loading_observation,
    "spaces": spaces.Box(low=0.0, high=1000.0, shape=(1,), dtype=np.float32),
}]

custom_observation, _ = PPTopoGym(custom_config).reset(options={"index": 4000})
print({key: value.shape for key, value in custom_observation.items()})

{'line_loadings': (41,), 'mean_loading': (1,)}


9. `info_observations` -- what lands in the info dict
------------------------------------------------------

Separately from the observation, every step reports a before/after pair for a list of
quantities in `info`. `info_observations` chooses that list; it defaults to bus voltages,
line loadings and the two overload aggregates.

Two of those aggregates -- `total_energy_overload` and `max_loading_percent` -- are
deliberately **not** in the observation registry. They are computable by name for `info`
and for the evaluation metrics, but putting them in the registry would grow the default
observation space and move the input dimension of every trained network.

In [17]:
print("info-only observations:", sorted(build_info_observation_registry()))

info_config = dict(narrow_config)
info_config["info_observations"] = ["max_loading_percent", "total_energy_overload"]

info_env = PPTopoGym(info_config)
info_env.reset(options={"index": 4000})
_, _, _, _, info = info_env.step(0)
print("info keys:", list(info))
print("max_loading_percent before/after:",
      info["max_loading_percent_before"], info["max_loading_percent_after"])

info-only observations: ['max_loading_percent', 'total_energy_overload']


info keys: ['current_step', 'profile_index', 'max_loading_percent_before', 'total_energy_overload_before', 'message', 'crashed', 'max_loading_percent_after', 'total_energy_overload_after', 'prev_actions', 'index_profile', '_source_instance_id']
max_loading_percent before/after: [55.52437] [55.72563]


10. Full reference
-------------------

| key | default | what it does |
|---|---|---|
| `net` | *required* | the pandapower grid (already expanded into substations) |
| `net_file` | -- | path to a grid on disk, used when `net` is not a `pandapowerNet` |
| `n_episodes` | *required* | number of episodes the timeseries is cut into |
| `episode_length` | *required* | timesteps per episode; must not exceed the timeseries length |
| `action_space` | *required* | list of action dicts; becomes `env.df_actions`, row 0 is DoNothing |
| `profiles` | `None` | absolute per-element timeseries; **assigned, not scaled**; wins over `net.profiles` |
| `resolution` | `1.0` | hours per timestep, used to turn overload power into energy |
| `pf_type` | `"ac"` | `"ac"` or `"dc"` power flow |
| `backend` | `"pandapower"` | `"lightsim"` solves a switch-free mirror net; ~10x faster, ~1e-11 agreement, opt-in |
| `nminus1` | `False` | evaluate single-outage contingencies each step; much slower |
| `n-1-topk` | `100.0` | only the top *k* % of lines by N-0 flow are switched off as contingencies |
| `n-1 parallel` | `False` | spread contingencies over worker processes; no-op inside child processes |
| `n-1 workers` | all CPUs | worker count; scale it with the contingency count, not the core count |
| `reward` | negative clipped max loading | callable, or a function name in `data/rewards.py` |
| `worst_reward` | `-1000.0` | reward returned when the power flow does not converge |
| `clip_max_loading` | `200.0` | loading ceiling in the default reward |
| `observation_keys` | all registry keys | which observations are emitted and declared |
| `observation` | `None` | extra custom observations: `{"name", "function", "spaces"}` |
| `fix_obs_space` | `True` | aggregate table observations to electrical nodes instead of table length |
| `static_obs_space` | `False` | declare node observations at their upper bound and zero-pad; needed for the vector envs |
| `info_observations` | voltages, loadings, overload aggregates | which before/after quantities land in `info` |

Two things that are **not** configuration keys but are asked about often:

* **lightsim2grid acceleration of the default backend** is not a key -- `use_ls2g="auto"` is
  the default of every power-flow entry point and is already active. It falls back silently
  to the native solver for DC power flow and for nets it cannot handle.
* **`reset` semantics** -- `seed` only seeds `random`; use `options={"index": N}` to reset to
  a specific timestep.

11. The shipped configurations
-------------------------------

`pandapower_env/data/example_configs.py` ships three ready-made configurations. All three
scale the grid until lines overload, expand every substation, generate the action list and
verify each action.

| function | grid | substations | what it adds |
|---|---|---|---|
| `config_case30()` | `case30` | all, double busbar | the default example: busbar switching and line switching. Takes `max_percent`, `overloaded_lines` and `init_scaling` to re-tune how congested the scaled grid is. |
| `config_30pst()` | `case30` | all, with a 3-busbar-with-PST substation at bus 5 | adds phase-shift-transformer tap actions (`trafos` / `tap_pos`) on top of the switching actions |
| `config_case89()` | `case89pegase` | all, double busbar | the large grid: 476 buses, 38 substations, 210 contingencies. Noticeably slower to build and to step. |

Building one is expensive, so build it once and hand `env.orig_config` to anything that
needs its own environment.

In [18]:
from pandapower_env.data.example_configs import config_30pst

pst_config = config_30pst()
pst_env = PPTopoGym(pst_config)
pst_actions = pst_env.df_actions
print("actions:", len(pst_actions))
pst_actions[pst_actions["trafos"].astype(bool)].head()

actions: 2401


,action,substations,states,open_switches,closed_switches,lines,disconnect_lines,trafos,tap_pos
2340,2340,[],[],[],[],[],[],[0],[-30.0]
2341,2341,[],[],[],[],[],[],[0],[-29.0]
2342,2342,[],[],[],[],[],[],[0],[-28.0]
2343,2343,[],[],[],[],[],[],[0],[-27.0]
2344,2344,[],[],[],[],[],[],[0],[-26.0]
